# Sentence Encoding and Semantic Clustering

This notebook shows how sentences from a historical Greek corpus are converted into numerical representations (**sentence embeddings**) and grouped according to semantic similarity. It is written so that readers without a technical background can follow the main analytical steps. The computational groups should be understood as **semantic groupings that support interpretation**, not as topics or ideological categories automatically discovered by an algorithm. Their substantive meaning is assessed by inspecting the sentences and recurrent lexical patterns within each group.

### Input
The notebook expects a sentence-level CSV file (here called `sentencized_corpus.csv`) containing the corpus text and relevant metadata such as chapter identifiers. Change the filename/path to match your local copy.



In [ ]:
import re
import numpy as np
import pandas as pd
from pprint import pprint

In [ ]:
#to avoid forking
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [ ]:
# Set the device to run on, e.g. GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
%pylab inline
df = pd.read_csv('writer_sentencized_corpus.csv')


In [ ]:
df.columns

In [ ]:
df.head(1)

In [ ]:
df.shape

In [ ]:
#just checking...
df['sents1'][2015]

In [ ]:
df.columns

In [ ]:
#unload not necessary columns
df = df[['Chapters', 'sents1', 'num_wds']]

In [ ]:
df.shape

In [ ]:
#assign the class and the label
df['Label'] = 2
df['Writer'] = 'Kordatos'


In [ ]:
df[1200:1250]

## 1. Encode the sentences

A sentence embedding is a numerical representation of a sentence. Sentences that are used in similar ways or express similar meanings tend to receive vectors that are closer together in the model's semantic space. This allows us to compare passages even when they do not use exactly the same words.


In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
from transformers import pipeline

In [ ]:
from sentence_transformers import SentenceTransformer,util

### Multilingual Sentence-BERT model

We use `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`, a multilingual SentenceTransformer model that represents each sentence as a 768-dimensional vector. The model supports Greek and is designed for semantic similarity tasks. The underlying Sentence-BERT approach was introduced by Reimers and Gurevych (2019). Instead of comparing every pair of sentences directly with a full transformer model, Sentence-BERT produces reusable sentence vectors that can be compared efficiently, commonly with cosine similarity.

Reference: Nils Reimers and Iryna Gurevych (2019), *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*, EMNLP-IJCNLP 2019.

Model: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`.


In [ ]:
# Load the multilingual SBERT model
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

# Test Greek sentences
greek_sentences = [
    "Αυτή είναι μια πρόταση.",
    "Αυτή είναι άλλη μία πρόταση.",
]

# Encode the sentences
sentence_embeddings = model.encode(greek_sentences)

In [ ]:
# Print the embeddings
for sentence, embedding in zip(greek_sentences, sentence_embeddings):
    print(f"Sentence: {sentence}")
    print(f"Embedding: {embedding}\n")

In [ ]:
#check the parameters
model

In [ ]:
#create a corpus to encode
corpus = list(df['sents'])


In [ ]:
#corpus_df = pd.DataFrame({'Chapters':  df['Chapters'], 'Sentences': corpus, 'Class': df['Class']}) #'Institute': df['Institute'
corpus_df = pd.DataFrame({'Chapters':  df['Chapters'], 'Sentences': corpus}) #'Institute': df['Institute'

In [ ]:
corpus_df.head()

The following cells encode the corpus and save the resulting sentence embeddings as a NumPy `.npy` file. Saving the numerical matrix separately preserves the embedding values exactly and avoids converting vectors into text. If the embeddings have already been generated, they can be loaded from the saved `.npy` file instead of being recomputed.


In [ ]:
#pass the corpus to the model
corpus_embeddings = model.encode(corpus_df['Sentences'].tolist(), normalize_embeddings=True, show_progress_bar=True)

In [ ]:
corpus_embeddings

In [ ]:
#save the embeddings
np.save('Writer_sents_embeddings.npy', corpus_embeddings)

In [ ]:
#insert the embeddings to check...
corpus_embeddings = np.load('Writer_sents_embeddings.npy')
#Kordatos_sents_no_accents_embeddings

In [ ]:
corpus_embeddings

In [ ]:
#looking good... 

In [ ]:
pd.set_option('display.width', None)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', None) # Display full content of each cell without truncation

## 2. From embeddings to broad semantic clusters

Each sentence embedding contains hundreds of numerical dimensions. Before clustering, we use **UMAP** to reduce these vectors to five dimensions. This makes the representation more manageable for the clustering step while attempting to preserve important local relationships among sentences. In this workflow, the five-dimensional UMAP representation is the input to HDBSCAN. A separate two-dimensional UMAP projection is created later only for visualization.


In [ ]:
#We reduce the dimensionality to 5 while keeping the size of the local neighborhood at 15
import umap

In [ ]:
# Test different n_neighbors values
neighbor_values = [5, 10, 15, 20, 30, 40, 50, 100]
avg_distances = []

for n in neighbor_values:
    umap_test = umap.UMAP(n_neighbors=n, n_components=5, metric="cosine", random_state=42)
    embedding_test = umap_test.fit_transform(corpus_embeddings)
    # Calculate mean pairwise distance in UMAP space
    avg_dist = np.mean(np.linalg.norm(embedding_test - embedding_test.mean(axis=0), axis=1))
    avg_distances.append(avg_dist)

In [ ]:
# Plot results
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(neighbor_values, avg_distances, marker="o", linestyle="-")
plt.xlabel("n_neighbors")
plt.ylabel("Average Distance in UMAP Space")
plt.title("Optimal n_neighbors Selection")
plt.show()

In [ ]:
#We reduce the dimensionality to 5 while keeping the size of the local neighborhood at 15
# Create a UMAP object with specific parameters
umap_model = umap.UMAP(
    n_neighbors=40,       # Focus on the local structure of the data (15 neighbors per point)
    n_components=5,       # Reduce the data to 5 dimensions
    metric='cosine',      # Use cosine similarity to measure distances between points
    random_state=42       # Ensure reproducibility with a fixed random seed
)

# Fit the UMAP model to the data and transform it
umap_embeddings = umap_model.fit_transform(corpus_embeddings)  # corpus_embeddings is your input data

In [ ]:
corpus_df['umap_embeddings'] = list(umap_embeddings)

### HDBSCAN clustering

We apply **HDBSCAN**, a density-based clustering algorithm, to the five-dimensional UMAP representation. Unlike methods that require a fixed number of clusters in advance, HDBSCAN identifies dense groups in the data and can initially mark observations that do not fit a dense group as noise (`-1`). The final model below uses `min_cluster_size=15` and `min_samples=2`. In a later step, sentences initially marked as noise are assigned to the nearest identified cluster. The resulting clusters are therefore broad semantic groupings rather than automatically identified historical topics.


In [ ]:
import hdbscan

In [ ]:
import numpy as np

# Define a range of min_cluster_size
min_cluster_sizes = [10, 20, 30, 40, 50, 60, 100]
for size in min_cluster_sizes:
    cluster_model = hdbscan.HDBSCAN(
        min_cluster_size=size,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    cluster = cluster_model.fit(umap_embeddings)
    print(f"Min Cluster Size: {size}, Number of Clusters: {len(np.unique(cluster.labels_))}")

In [ ]:
# we apply the clustering technique on the embeddings after being reduced by umap

# Step 1: Create an HDBSCAN clustering object with specific parameters
cluster_model = hdbscan.HDBSCAN(
    min_cluster_size=15,            # Minimum number of points required to form a cluster
    min_samples=2,                  # Minimum number of samples in a point's neighborhood for it to be a core point
    metric='euclidean',             # Use Euclidean distance to measure distances between points
    cluster_selection_method='eom'  # eom= "Excess of Mass" method to find clusters (more stable for varying densities)
)

# Step 2: Fit the clustering model to the UMAP-reduced data
cluster = cluster_model.fit(umap_embeddings)  # umap_embeddings is the input data to be clustered

In [ ]:
corpus_df['Cluster'] = cluster_model.labels_

In [ ]:
# Count the number of points in each cluster
cluster_sizes = corpus_df['Cluster'].value_counts()

# Print the cluster sizes
cluster_sizes

In [ ]:
# Get the cluster labels
labels = cluster.labels_

# Count the unique clusters (excluding outliers labeled as -1)
num_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"Number of clusters (excluding outliers): {num_clusters}")

### Reimport some outliers into clusters (Optional)

In [ ]:
from collections import Counter

# Count the points in each cluster
cluster_counts = Counter(labels)

# Print the cluster sizes
print("Cluster sizes:")
for cluster_id, count in cluster_counts.items():
    if cluster_id != -1:  # Exclude outliers
        print(f"Cluster {cluster_id}: {count} points")

In [ ]:
corpus_df.tail()

In [ ]:
# Filter for sentences in the `-1` cluster
outliers = corpus_df[corpus_df['Cluster'] == -1]
print(f"Number of sentences in the -1 cluster: {len(outliers)}")

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Get core points
core_points = corpus_df[corpus_df['Cluster'] != -1]
core_embeddings = np.vstack(core_points['umap_embeddings'])
core_labels = core_points['Cluster'].values

# For each outlier, find nearest core point
nn = NearestNeighbors(n_neighbors=1, metric='cosine').fit(core_embeddings)
outlier_embeddings = np.vstack(corpus_df[corpus_df['Cluster'] == -1]['umap_embeddings'])
distances, indices = nn.kneighbors(outlier_embeddings)

# Assign the nearest cluster label
nearest_clusters = core_labels[indices.flatten()]
corpus_df.loc[corpus_df['Cluster'] == -1, 'Updated_Cluster'] = nearest_clusters

In [ ]:
corpus_df

In [ ]:
corpus_df['Updated_Cluster'] = corpus_df['Updated_Cluster'].fillna(corpus_df['Cluster'])
corpus_df['Updated_Cluster'] = corpus_df['Updated_Cluster'].astype(int)


In [ ]:
corpus_df

In [ ]:
updated_outliers_count = (corpus_df['Updated_Cluster'] == -1).sum()
print(f"Remaining sentences in the -1 cluster: {updated_outliers_count}")

In [ ]:
remaining_outliers = corpus_df[corpus_df['Cluster'] == -1]
print(f"Remaining sentences originally marked as outliers: {len(remaining_outliers)}")

# If you also want to check if any are *still* unassigned in Updated_Cluster
still_unassigned = corpus_df[corpus_df['Updated_Cluster'] == -1]
print(f"Sentences still unassigned after update: {len(still_unassigned)}")

In [ ]:
from collections import Counter

updated_cluster_counts = Counter(corpus_df['Updated_Cluster'])
print("Updated cluster distribution:")
print(updated_cluster_counts)

In [ ]:
# Example: Inspect reassigned sentences
reassigned_sentences = corpus_df[(corpus_df['Cluster'] == -1) & (corpus_df['Updated_Cluster'] != -1)]
reassigned_sentences.head(10)

### Visualizing the clusters

For visual inspection, the original sentence embeddings are projected separately into two dimensions with UMAP. This 2D map helps us see the broad arrangement of sentences and clusters, but it does **not** determine cluster membership. The actual HDBSCAN clustering above was performed on the five-dimensional UMAP representation.


In [ ]:
# organize data in a dataframe including the labels of the clusters. Labels are just numbers assigned by teh algorithm to the clusters. 
#They are mathematical concepts and dont have a semantic meaning.

umap_data = umap.UMAP(n_neighbors=40, n_components=2, #further lower the dimensioality to 2 so we can see the clusters in 2 dimensions
                      min_dist=0.1, metric='cosine').fit_transform(corpus_embeddings)

In [ ]:
#Create a DataFrame with the 2D coordinates and cluster labels
result = pd.DataFrame(umap_data, columns=['x', 'y'])
result['labels'] = corpus_df['Updated_Cluster']  # Use updated cluster labels if applicable


In [ ]:
# visualize
import matplotlib.pyplot as plt

In [ ]:
# scatter plot
fig, ax = plt.subplots(figsize=(12, 12))

# Separate outliers and clustered points
outliers = result[result['labels'] == -1]
clustered = result[result['labels'] != -1]

# Plot the outliers in grey
plt.scatter(outliers['x'], outliers['y'], color='#BDBDBD', s=8, alpha=0.5, label='Outliers')

# Plot the clustered points with a colormap
scatter = plt.scatter(
    clustered['x'], clustered['y'], c=clustered['labels'], cmap='hsv_r', s=2, alpha=0.7
)

# Add colorbar and labels
plt.colorbar(scatter, label='Cluster Labels')
plt.title("UMAP Projection with HDBSCAN Clusters")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.legend()
plt.show()

## 3. Interpreting the semantic clusters

Clustering gives each sentence a numerical cluster label, but the number itself has no substantive meaning. The following steps help us inspect what distinguishes one cluster from another. We use **class-based TF-IDF (c-TF-IDF)** to identify words and expressions that are especially characteristic of each cluster. These lexical cues support interpretation; they should not be treated as automatically discovered topics.


### Characteristic vocabulary with c-TF-IDF

c-TF-IDF treats all sentences in a cluster as a combined group and asks which words are especially characteristic of that group compared with the other clusters. In simple terms, a word receives a higher score when it is frequent in one cluster but less common across the remaining clusters. This provides a compact lexical summary that can be read alongside representative sentences.


In [ ]:
# Group sentences by Updated_Cluster
docs_per_cluster = corpus_df.groupby('Updated_Cluster', as_index=False).agg({'Sentences': '/ '.join})


In [ ]:
docs_per_cluster.shape

In [ ]:
pd.set_option('display.width', 50)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', 1000)

In [ ]:
docs_per_cluster.head(1)

### LLM-assisted cluster labels

The following section uses an LLM to generate concise provisional labels for the semantic clusters. These labels are interpretive aids rather than outputs of the clustering algorithm itself. They should be checked against the underlying sentences and characteristic vocabulary before being used substantively.


In [ ]:
import os
import openai
from openai import OpenAI
client = OpenAI()

In [ ]:
# Define the function using `client.chat.completions.create`
def generate_cluster_title(row, client, model="gpt-4o"):
    sentences = row['Sentences']
    index = row.name
    try:
        # Call the OpenAI API
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a historical discourse analyst specializing in language ideologies. "
                        "You will receive a cluster of Greek sentences. "
                        "Your task is to generate a short, vivid, and thematically precise English title (max 10 words) "
                        "that captures not only the topic but also the tone or emotional charge of the text. "
                        "Vary your language. Avoid repeating 'struggle' or 'identity'. "
                        "You may use historical, satirical, academic or polemical styles as well as names depending on the text. "
                    "Be specific. Use powerful or metaphorical phrasing when appropriate. Avoid clichés. No punctuation."


                    )
                },
                {
                    "role": "user",
                    "content": sentences
                }
            ],
            max_tokens=100,
            temperature=0.65
        )

        full_response = response.choices[0].message.content.strip()
        print(f" Row {index} processed: {full_response}")
        return full_response

    except Exception as e:
        error_msg = f"[ERROR @ row {index}]: {str(e)}"
        print(error_msg)
        return error_msg

In [ ]:
# Apply the function row by row and assign new columns
docs_per_cluster["Cluster_title"] = docs_per_cluster.apply(lambda row: generate_cluster_title(row, client), axis=1)


In [ ]:
pd.set_option('display.width', 50)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', 100)


In [ ]:
docs_per_cluster

Now, let's apply the class-based TF-IDF, by joining documents within a class. Where the frequency of each word 't' is extracted for each class i and divided by the total number of words w. This action can be seen as a form of regularization of frequent words in the class. Next, the total, unjoined, number of documents m is divided by the total frequency of word t across all classes n.

### Apply c_tf_idf

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

In [ ]:
# Function to calculate class-based TF-IDF
def c_tf_idf(documents, m, ngram_range=(1, 3)):  # Adjust ngram_range for granularity
    #vectorizer = CountVectorizer(ngram_range=ngram_range, stop_words=cleaned_stop_words).fit(documents)
    #vectorizer = CountVectorizer(ngram_range=ngram_range, stop_words=list(cleaned_stop_words)).fit(documents)
    vectorizer = CountVectorizer(ngram_range=ngram_range, stop_words=list(cleaned_stop_words), min_df=2,  # Ignore very rare terms
    max_df=0.8  # Ignore very frequent terms
).fit(documents)
    t = vectorizer.transform(documents).toarray()
    w = t.sum(axis=1)
    tf = np.divide(t.T, w, out=np.zeros_like(t.T, dtype=float), where=w != 0)
    sum_t = t.sum(axis=0)
    idf = np.log(np.divide(m, sum_t, out=np.zeros_like(sum_t, dtype=float), where=sum_t != 0)).reshape(-1, 1)
    tf_idf = np.multiply(tf, idf)
    return tf_idf, vectorizer

In [ ]:
# Calculate TF-IDF
tf_idf, vectorizer = c_tf_idf(docs_per_cluster['Sentences'], m=len(corpus_df))

In order to create a topic representation, we take the top 20 words per topic based on their c-TF-IDF scores. The higher the score, the more representative it should be of its topic. 
In the following, we add to the function  an additional argument top_n_words, which is the dictionary of top keywords per topic that we obtain from the extract_top_n_words_per_topic function. The function then adds a new column to the topic_sizes dataframe called 'Top_Keywords', which contains the top keywords as a comma-separated string for each topic. The lambda function extracts the top keywords from the top_n_words dictionary based on the topic value and joins them using a comma separator. The function returns the updated topic_sizes dataframe.

In [ ]:
# Extract top keywords
def extract_top_n_words_per_cluster(tf_idf, vectorizer, docs_per_cluster, n=30):
    words = vectorizer.get_feature_names_out()
    labels = docs_per_cluster['Updated_Cluster']
    tf_idf_transposed = tf_idf.T
    indices = tf_idf_transposed.argsort()[:, -n:]
    top_n_words = {
        label: [(words[j], tf_idf_transposed[i][j]) for j in indices[i]][::-1]
        for i, label in enumerate(labels)
    }
    return top_n_words

top_keywords = extract_top_n_words_per_cluster(tf_idf, vectorizer, docs_per_cluster, n=30)

In order to create a topic representation, we take the top 20 words per topic based on their c-TF-IDF scores. The higher the score, the more representative it should be of its topic. 
In the following, we add to the function  an additional argument top_n_words, which is the dictionary of top keywords per topic that we obtain from the extract_top_n_words_per_topic function. The function then adds a new column to the topic_sizes dataframe called 'Top_Keywords', which contains the top keywords as a comma-separated string for each topic. The lambda function extracts the top keywords from the top_n_words dictionary based on the topic value and joins them using a comma separator. The function returns the updated topic_sizes dataframe.

In [ ]:
# check
top_keywords

In [ ]:
# Extract cluster sizes
def extract_cluster_sizes(df, top_keywords):
    cluster_sizes = (corpus_df.groupby(['Updated_Cluster'])
                       .Sentences
                       .count()
                       .reset_index()
                       .rename({"Updated_Cluster": "Updated_Cluster", "Sentences": "Size"}, axis='columns')
                       .sort_values("Size", ascending=False))
    cluster_sizes['Top_Keywords'] = cluster_sizes['Updated_Cluster'].apply(lambda x: ', '.join([word[0] for word in top_keywords[x]]))
    return cluster_sizes

# Get cluster sizes with top keywords
cluster_sizes = extract_cluster_sizes(corpus_df, top_keywords)

# Display the results
pd.set_option('display.max_colwidth', None)

In [ ]:
cluster_sizes.head(50)

In [ ]:
cluster_sizes.shape

In [ ]:
docs_per_cluster.shape

In [ ]:
cluster_sizes.columns

In [ ]:
docs_per_cluster.columns

In [ ]:
#Merge cluster_sizes and docs_per_cluster on 'Updated_Cluster'
cluster_sizes = pd.merge(docs_per_cluster, cluster_sizes, on='Updated_Cluster', how='left')

In [ ]:
pd.set_option('display.width', 50)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', 100)

In [ ]:
#check
cluster_sizes

In [ ]:
cluster_sizes[1:2]

### Researcher review of cluster labels

Some automatically generated labels are edited after inspection of the underlying sentences. This step reflects the distinction between computational grouping and historical interpretation: the algorithm groups semantically related passages, while the researcher determines how those groupings should be described.


In [ ]:
# examples 
cluster_sizes.loc[cluster_sizes['Updated_Cluster'] == 1, 'Cluster_title2'] = "Generic"
cluster_sizes.loc[cluster_sizes['Updated_Cluster'] == 4, 'Cluster_title2'] = "Language as Class War: The Social Politics of Greek"
cluster_sizes.loc[cluster_sizes['Updated_Cluster'] == 9, 'Cluster_title2'] = "Clergy, class and demotic spread: fanaticism, propaganda and betrayals (Psicharis, Volos, Oresteiaka)"



In [ ]:
# check
cluster_sizes

### Merge cluster information back into the sentence-level corpus

The interpreted cluster information is now joined back to the sentence-level dataframe so that each sentence retains its cluster assignment together with the relevant cluster-level metadata.


In [ ]:
cluster_sizes.rename(columns={'Sentences': 'Clustered_Sentences'}, inplace=True)

In [ ]:
cluster_sizes

In [ ]:
corpus_df.head(2)

In [ ]:
#lets drop the ogiginal cluster column
corpus_df.drop('Cluster', axis=1, inplace=True)
corpus_df.head(2)

In [ ]:
# insert the name of the writer e.g.
corpus_df['Writer'] = 'Kordatos'

In [ ]:
# Merge cluster sizes and top keywords back into corpus_df
corpus_df = corpus_df.merge(cluster_sizes, on='Updated_Cluster', how='left')

In [ ]:
pd.set_option('display.width', 50)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', 50)

In [ ]:
corpus_df

In [ ]:
# Save the sentence-level dataset in Parquet format.
# Unlike CSV, Parquet can preserve nested numerical data such as the UMAP vectors
# without turning them into plain text strings. The original sentence embeddings
# are also retained separately in Writer_sents_embeddings.npy (saved above).

corpus_df_to_save = corpus_df.copy()
corpus_df_to_save['umap_embeddings'] = corpus_df_to_save['umap_embeddings'].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)
corpus_df_to_save.to_parquet('Writer_Embeddings_Clusters.parquet', index=False)


#### Why Parquet rather than CSV?
The dataframe contains a column of UMAP vectors. A CSV file is a plain-text format, so arrays stored in a dataframe cell are written out as text and are not automatically restored as numerical vectors when the file is read again. Parquet supports structured data and is therefore safer for preserving this kind of numerical output. The full 768-dimensional SentenceTransformer embeddings are saved separately as `Writer_sents_embeddings.npy`, which preserves the NumPy array directly. The Parquet file stores the sentence-level corpus, cluster assignments, metadata, and reduced UMAP vectors.
